## spacing_z resampling

resampling : (1.0, 1.0, 2.0)

- covid의 z 방향 slice 간격을 2.0mm 기준으로 다시 샘플링
- 나머지 source는 1.0으로 샘플링
- slice 수는 증가
- 하지만 실제 해상도가 진짜 좋아지는 건 아니고 보간된 slice가 생김

In [1]:
import SimpleITK as sitk

In [9]:
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd
from scipy.ndimage import zoom

RAW_ROOT = Path(r"C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\raw_data")
VOLUME_ROOT = RAW_ROOT / "volumes"
LABEL_ROOT = RAW_ROOT / "labels"

OUTPUT_ROOT = Path(r"C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\t11_t12_resampled_1x1x2")
OUTPUT_IMAGE_ROOT = OUTPUT_ROOT / "images"
OUTPUT_LABEL_ROOT = OUTPUT_ROOT / "labels"

TARGET_SPACING = (1.0, 1.0, 2.0)

T11_LABEL = 18
T12_LABEL = 19

OUTPUT_IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_LABEL_ROOT.mkdir(parents=True, exist_ok=True)

In [10]:
def find_label_path(volume_path):
    rel = volume_path.relative_to(VOLUME_ROOT)
    label_dir = LABEL_ROOT / rel.parent
    stem = volume_path.name.replace(".nii.gz", "")

    candidates = [
        label_dir / f"{stem}_seg.nii.gz",
        label_dir / volume_path.name,
        label_dir / f"{stem}.nii.gz",
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    return None


def make_new_affine(old_affine, target_spacing):
    new_affine = old_affine.copy()

    for axis in range(3):
        direction = old_affine[:3, axis]
        norm = np.linalg.norm(direction)

        if norm > 0:
            new_affine[:3, axis] = direction / norm * target_spacing[axis]

    return new_affine


def resample_array(arr, original_spacing, target_spacing, order):
    zoom_factors = [
        original_spacing[i] / target_spacing[i]
        for i in range(3)
    ]

    return zoom(arr, zoom=zoom_factors, order=order)

In [11]:
def resample_sitk_image(
    sitk_image,
    target_spacing,
    interpolator,
    default_value=0,
):
    original_spacing = sitk_image.GetSpacing()
    original_size = sitk_image.GetSize()

    target_size = [
        int(round(original_size[i] * (original_spacing[i] / target_spacing[i])))
        for i in range(3)
    ]

    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(target_spacing)
    resampler.SetSize(target_size)
    resampler.SetOutputDirection(sitk_image.GetDirection())
    resampler.SetOutputOrigin(sitk_image.GetOrigin())
    resampler.SetTransform(sitk.Transform())
    resampler.SetDefaultPixelValue(default_value)
    resampler.SetInterpolator(interpolator)

    return resampler.Execute(sitk_image)

#### 주의: SimpleITK array shape은 (z, y, x)이고, image spacing은 (x, y, z)입니다. 위 코드는 SimpleITK 변환 규칙에 맞게 작동합니다.

In [12]:
def make_t11_t12_label(label_image):
    label_arr = sitk.GetArrayFromImage(label_image)

    target_arr = np.zeros_like(label_arr, dtype=np.uint8)
    target_arr[label_arr == T11_LABEL] = 1
    target_arr[label_arr == T12_LABEL] = 2

    target_image = sitk.GetImageFromArray(target_arr)
    target_image.CopyInformation(label_image)

    return target_image

In [13]:
df_t11_t12_final = pd.read_pickle("df_t11_t12_final_candidates.pkl")

print("Final candidates:", len(df_t11_t12_final))
display(df_t11_t12_final.groupby("source").size())

Final candidates: 939


source
COLONOG     752
COVID-19     37
MSD-T10     150
dtype: int64

In [17]:
df_t11_t12_final = pd.read_pickle("df_t11_t12_final_candidates.pkl")

MAX_CASES = None  # 테스트 후 None으로 변경

target_df = df_t11_t12_final.copy()

if MAX_CASES is not None:
    target_df = target_df.head(MAX_CASES)

rows = []

for idx, row in enumerate(target_df.itertuples(index=False), start=1):
    source = row.source
    file_name = row.file_name

    volume_path = VOLUME_ROOT / source / file_name
    label_path = find_label_path(volume_path)

    if label_path is None:
        print("Label not found:", file_name)
        continue

    img_nii = nib.load(volume_path)
    seg_nii = nib.load(label_path)

    image = np.asanyarray(img_nii.dataobj).astype(np.float32)
    seg = np.asanyarray(seg_nii.dataobj)

    original_spacing = img_nii.header.get_zooms()[:3]

    target_seg = np.zeros_like(seg, dtype=np.uint8)
    target_seg[seg == T11_LABEL] = 1
    target_seg[seg == T12_LABEL] = 2

    image_resampled = resample_array(
        image,
        original_spacing=original_spacing,
        target_spacing=TARGET_SPACING,
        order=1,
    ).astype(np.float32)

    label_resampled = resample_array(
        target_seg,
        original_spacing=original_spacing,
        target_spacing=TARGET_SPACING,
        order=0,
    ).astype(np.uint8)

    new_affine = make_new_affine(img_nii.affine, TARGET_SPACING)

    source_image_dir = OUTPUT_IMAGE_ROOT / source
    source_label_dir = OUTPUT_LABEL_ROOT / source

    source_image_dir.mkdir(parents=True, exist_ok=True)
    source_label_dir.mkdir(parents=True, exist_ok=True)

    output_image_path = source_image_dir / file_name
    output_label_path = source_label_dir / file_name.replace(".nii.gz", "_t11_t12.nii.gz")

    nib.save(
        nib.Nifti1Image(image_resampled, new_affine),
        output_image_path,
    )

    nib.save(
        nib.Nifti1Image(label_resampled, new_affine),
        output_label_path,
    )

    rows.append({
        "source": source,
        "patient_id": row.patient_id,
        "file_name": file_name,
        "input_image_path": str(volume_path),
        "input_label_path": str(label_path),
        "output_image_path": str(output_image_path),
        "output_label_path": str(output_label_path),
        "original_spacing": original_spacing,
        "target_spacing": TARGET_SPACING,
        "original_shape": image.shape,
        "resampled_shape": image_resampled.shape,
        "label_values": tuple(np.unique(label_resampled)),
    })

    print(f"[{idx}/{len(target_df)}] Saved:", file_name)

df_resample_log = pd.DataFrame(rows)
display(df_resample_log)

[1/939] Saved: 1.3.6.1.4.1.9328.50.4.0001.nii.gz
[2/939] Saved: 1.3.6.1.4.1.9328.50.4.0002.nii.gz
[3/939] Saved: 1.3.6.1.4.1.9328.50.4.0003.nii.gz
[4/939] Saved: 1.3.6.1.4.1.9328.50.4.0004.nii.gz
[5/939] Saved: 1.3.6.1.4.1.9328.50.4.0005.nii.gz
[6/939] Saved: 1.3.6.1.4.1.9328.50.4.0006.nii.gz
[7/939] Saved: 1.3.6.1.4.1.9328.50.4.0007.nii.gz
[8/939] Saved: 1.3.6.1.4.1.9328.50.4.0008.nii.gz
[9/939] Saved: 1.3.6.1.4.1.9328.50.4.0009.nii.gz
[10/939] Saved: 1.3.6.1.4.1.9328.50.4.0010.nii.gz
[11/939] Saved: 1.3.6.1.4.1.9328.50.4.0011.nii.gz
[12/939] Saved: 1.3.6.1.4.1.9328.50.4.0012.nii.gz
[13/939] Saved: 1.3.6.1.4.1.9328.50.4.0013.nii.gz
[14/939] Saved: 1.3.6.1.4.1.9328.50.4.0014.nii.gz
[15/939] Saved: 1.3.6.1.4.1.9328.50.4.0015.nii.gz
[16/939] Saved: 1.3.6.1.4.1.9328.50.4.0016.nii.gz
[17/939] Saved: 1.3.6.1.4.1.9328.50.4.0017.nii.gz
[18/939] Saved: 1.3.6.1.4.1.9328.50.4.0018.nii.gz
[19/939] Saved: 1.3.6.1.4.1.9328.50.4.0020.nii.gz
[20/939] Saved: 1.3.6.1.4.1.9328.50.4.0021.nii.gz
[21/939] 

,source,patient_id,file_name,input_image_path,input_label_path,output_image_path,output_label_path,original_spacing,target_spacing,original_shape,resampled_shape,label_values
0,COLONOG,1.3.6.1.4.1.9328.50.4.0001,1.3.6.1.4.1.9328.50.4.0001.nii.gz,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,"(0.78125, 0.78125, 0.8)","(1.0, 1.0, 2.0)","(512, 512, 604)","(400, 400, 242)","(0, 1, 2)"
1,COLONOG,1.3.6.1.4.1.9328.50.4.0002,1.3.6.1.4.1.9328.50.4.0002.nii.gz,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,"(0.78125, 0.78125, 0.8)","(1.0, 1.0, 2.0)","(512, 512, 637)","(400, 400, 255)","(0, 1, 2)"
2,COLONOG,1.3.6.1.4.1.9328.50.4.0003,1.3.6.1.4.1.9328.50.4.0003.nii.gz,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,"(0.78125, 0.78125, 0.8)","(1.0, 1.0, 2.0)","(512, 512, 663)","(400, 400, 265)","(0, 1, 2)"
3,COLONOG,1.3.6.1.4.1.9328.50.4.0004,1.3.6.1.4.1.9328.50.4.0004.nii.gz,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,"(0.78125, 0.78125, 0.8)","(1.0, 1.0, 2.0)","(512, 512, 520)","(400, 400, 208)","(0, 1, 2)"
4,COLONOG,1.3.6.1.4.1.9328.50.4.0005,1.3.6.1.4.1.9328.50.4.0005.nii.gz,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,"(0.60546875, 0.60546875, 1.0)","(1.0, 1.0, 2.0)","(512, 512, 401)","(310, 310, 200)","(0, 1, 2)"
...,...,...,...,...,...,...,...,...,...,...,...,...
934,MSD-T10,liver_95,liver_95.nii.gz,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,"(0.738, 0.738, 0.8)","(1.0, 1.0, 2.0)","(512, 512, 841)","(378, 378, 336)","(0, 1, 2)"
935,MSD-T10,liver_96,liver_96.nii.gz,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,"(0.74609375, 0.74609375, 0.7)","(1.0, 1.0, 2.0)","(512, 512, 722)","(382, 382, 253)","(0, 1, 2)"
936,MSD-T10,liver_97,liver_97.nii.gz,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,"(0.72265625, 0.72265625, 0.7)","(1.0, 1.0, 2.0)","(512, 512, 671)","(370, 370, 235)","(0, 1, 2)"
937,MSD-T10,liver_98,liver_98.nii.gz,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,"(0.734375, 0.734375, 0.7)","(1.0, 1.0, 2.0)","(512, 512, 645)","(376, 376, 226)","(0, 1, 2)"


In [15]:
df_resample_log[["source", "file_name", "original_spacing", "target_spacing", "original_shape", "resampled_shape", "label_values"]]

,source,file_name,original_spacing,target_spacing,original_shape,resampled_shape,label_values
0,COLONOG,1.3.6.1.4.1.9328.50.4.0001.nii.gz,"(0.78125, 0.78125, 0.8)","(1.0, 1.0, 2.0)","(512, 512, 604)","(400, 400, 242)","(0, 1, 2)"
1,COLONOG,1.3.6.1.4.1.9328.50.4.0002.nii.gz,"(0.78125, 0.78125, 0.8)","(1.0, 1.0, 2.0)","(512, 512, 637)","(400, 400, 255)","(0, 1, 2)"
2,COLONOG,1.3.6.1.4.1.9328.50.4.0003.nii.gz,"(0.78125, 0.78125, 0.8)","(1.0, 1.0, 2.0)","(512, 512, 663)","(400, 400, 265)","(0, 1, 2)"


In [16]:
for _, row in df_resample_log.iterrows():
    img = nib.load(row["output_image_path"])
    seg = nib.load(row["output_label_path"])

    print(row["file_name"])
    print("image shape:", img.shape)
    print("label shape:", seg.shape)
    print("same shape:", img.shape == seg.shape)
    print("label values:", np.unique(np.asanyarray(seg.dataobj)))
    print()

1.3.6.1.4.1.9328.50.4.0001.nii.gz
image shape: (400, 400, 242)
label shape: (400, 400, 242)
same shape: True
label values: [0 1 2]

1.3.6.1.4.1.9328.50.4.0002.nii.gz
image shape: (400, 400, 255)
label shape: (400, 400, 255)
same shape: True
label values: [0 1 2]

1.3.6.1.4.1.9328.50.4.0003.nii.gz
image shape: (400, 400, 265)
label shape: (400, 400, 265)
same shape: True
label values: [0 1 2]



In [21]:
from pathlib import Path
import nibabel as nib
import numpy as np
import pandas as pd

OUTPUT_ROOT = Path(r"C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\t11_t12_resampled_1x1x2")
OUTPUT_IMAGE_ROOT = OUTPUT_ROOT / "images"
OUTPUT_LABEL_ROOT = OUTPUT_ROOT / "labels"

rows = []

for image_path in sorted(OUTPUT_IMAGE_ROOT.rglob("*.nii.gz")):
    source = image_path.parent.name
    file_name = image_path.name

    label_path = OUTPUT_LABEL_ROOT / source / file_name.replace(".nii.gz", "_t11_t12.nii.gz")

    image_exists = image_path.exists()
    label_exists = label_path.exists()

    if not label_exists:
        rows.append({
            "source": source,
            "file_name": file_name,
            "image_exists": image_exists,
            "label_exists": False,
        })
        continue

    img = nib.load(image_path)
    seg = nib.load(label_path)

    label_values = tuple(np.unique(np.asanyarray(seg.dataobj)).astype(int))

    rows.append({
        "source": source,
        "file_name": file_name,
        "image_path": image_path,
        "label_path": label_path,
        "image_exists": image_exists,
        "label_exists": label_exists,
        "image_shape": img.shape,
        "label_shape": seg.shape,
        "same_shape": img.shape == seg.shape,
        "image_spacing": img.header.get_zooms()[:3],
        "label_spacing": seg.header.get_zooms()[:3],
        "label_values": label_values,
        "has_T11": 1 in label_values,
        "has_T12": 2 in label_values,
        "t11_voxels": int(np.sum(np.asanyarray(seg.dataobj) == 1)),
        "t12_voxels": int(np.sum(np.asanyarray(seg.dataobj) == 2)),
    })

df_resampled_check = pd.DataFrame(rows)

display(df_resampled_check.head())
print("Checked:", len(df_resampled_check))

,source,file_name,image_path,label_path,image_exists,label_exists,image_shape,label_shape,same_shape,image_spacing,label_spacing,label_values,has_T11,has_T12,t11_voxels,t12_voxels
0,COLONOG,1.3.6.1.4.1.9328.50.4.0001.nii.gz,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,True,True,"(400, 400, 242)","(400, 400, 242)",True,"(1.0, 1.0, 2.0)","(1.0, 1.0, 2.0)","(0, 1, 2)",True,True,16555,19356
1,COLONOG,1.3.6.1.4.1.9328.50.4.0002.nii.gz,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,True,True,"(400, 400, 255)","(400, 400, 255)",True,"(1.0, 1.0, 2.0)","(1.0, 1.0, 2.0)","(0, 1, 2)",True,True,29404,32622
2,COLONOG,1.3.6.1.4.1.9328.50.4.0003.nii.gz,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,True,True,"(400, 400, 265)","(400, 400, 265)",True,"(1.0, 1.0, 2.0)","(1.0, 1.0, 2.0)","(0, 1, 2)",True,True,23154,26119
3,COLONOG,1.3.6.1.4.1.9328.50.4.0004.nii.gz,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,True,True,"(400, 400, 208)","(400, 400, 208)",True,"(1.0, 1.0, 2.0)","(1.0, 1.0, 2.0)","(0, 1, 2)",True,True,20115,21226
4,COLONOG,1.3.6.1.4.1.9328.50.4.0005.nii.gz,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,C:\Users\comds\Desktop\졸업작품\dataset\CTSpine1K\...,True,True,"(310, 310, 200)","(310, 310, 200)",True,"(1.0, 1.0, 2.0)","(1.0, 1.0, 2.0)","(0, 1, 2)",True,True,26675,28035


Checked: 939


In [22]:
print("Total:", len(df_resampled_check))
print("Missing labels:", (~df_resampled_check["label_exists"]).sum())
print("Shape mismatch:", (~df_resampled_check["same_shape"]).sum())
print("Missing T11:", (~df_resampled_check["has_T11"]).sum())
print("Missing T12:", (~df_resampled_check["has_T12"]).sum())

display(df_resampled_check["label_values"].value_counts())

Total: 939
Missing labels: 0
Shape mismatch: 0
Missing T11: 0
Missing T12: 0


label_values
(0, 1, 2)    939
Name: count, dtype: int64

In [23]:
display(
    df_resampled_check
    .groupby("source")
    .size()
    .rename("n_resampled")
    .to_frame()
)

,n_resampled
source,
COLONOG,752
COVID-19,37
MSD-T10,150


In [24]:
display(
    df_resampled_check[["image_spacing", "label_spacing"]]
    .head()
)

,image_spacing,label_spacing
0,"(1.0, 1.0, 2.0)","(1.0, 1.0, 2.0)"
1,"(1.0, 1.0, 2.0)","(1.0, 1.0, 2.0)"
2,"(1.0, 1.0, 2.0)","(1.0, 1.0, 2.0)"
3,"(1.0, 1.0, 2.0)","(1.0, 1.0, 2.0)"
4,"(1.0, 1.0, 2.0)","(1.0, 1.0, 2.0)"


In [25]:
problem_cases = df_resampled_check[
    (~df_resampled_check["label_exists"]) |
    (~df_resampled_check["same_shape"]) |
    (~df_resampled_check["has_T11"]) |
    (~df_resampled_check["has_T12"])
]

display(problem_cases)

,source,file_name,image_path,label_path,image_exists,label_exists,image_shape,label_shape,same_shape,image_spacing,label_spacing,label_values,has_T11,has_T12,t11_voxels,t12_voxels


In [26]:
df_resampled_check.to_csv(
    OUTPUT_ROOT / "resampled_sanity_check.csv",
    index=False,
    encoding="utf-8-sig",
)

df_resampled_check.to_pickle(
    OUTPUT_ROOT / "resampled_sanity_check.pkl"
)

## segmentation학습 전 crop 수행 해야됨